# Бренд, артикул, партномер, тип (товара)


## 1. Импорты


In [41]:
import pandas as pd
import numpy as np
import json
import re
from rapidfuzz import fuzz


## 2. Загрузка пар


In [42]:
pairs = pd.read_parquet('data/pairs.parquet', columns=['id1', 'id2', 'name_1', 'name_2', 'target', 'attributes_1', 'attributes_2'])

In [43]:
pairs.head()

,id1,id2,name_1,name_2,target,attributes_1,attributes_2
0,32,188978721454,очки с футляром-змейка fedrov мод. 1047 цвет 3...,фотохромные очки с футляром-змейка fedrov мод....,0.0,"{""диоптрии"":""- 2.00"",""цвет оправы"":""золотистый...","{""наличие флексов"":""да"",""затемнение"":""коричнев..."
1,34,197568543106,"зубная щетка r.o.c.s. black edition classic, ч...","зубная щетка r.o.c.s. red edition classic, сре...",0.0,"{""бренд"":""r.o.c.s."",""тип"":""зубная щетка"",""уров...","{""целевая аудитория"":""взрослая"",""бренд"":""r.o.c..."
2,37,661425040668,сандалии fre gamo,fre gamo / сандалии,0.0,"{""бренд в одежде и обуви"":""fre gamo"",""страна б...","{""цвет"":""желтый"",""страна производства"":""китай""..."
3,97,274877917548,acuvue астигматические линзы oasys for astigma...,acuvue / контактные линзы acuvue oasys for ast...,0.0,"{""ось (axis)"":""10"",""бренд"":""acuvue"",""цилиндр (...","{""состав"":""силикон-гидрогель"",""тип линз"":""тори..."
4,154,283467899529,"дворцовый пуэр (5 лет), 500 г",чайная линия ореховый пуэр (500 гр) вес: 500 гр,0.0,"{""бренд"":""weiserhouse"",""условия хранения"":""хра...","{""бренд"":""чайная линия"",""вид чая"":""пуэр"",""вкус..."


## 3. Очистка и извлечение атрибутов


In [44]:
no_article = ['нет', '-', '0', 'null', '_']
no_brand = ['нет бренда', 'не определен', 'noname', 'nobrand', 'без бренда', 'нет', 'нет бренд', '---','неизвестен', '.', 'туалетная вода', 'кеды', 'мягкая игрушка', 'производитель', 'оверсайз', 'no name']

def clean(value, bad_values, min_len=1):
    if value is None:
        return None
    value = value.lower().strip()
    if value == '' or value in bad_values or len(value) < min_len:
        return None
    return value



def parse_attributes(attribute):
    attribute = json.loads(attribute)
    brand_1 = attribute.get('бренд')
    brand_2 = attribute.get('бренд в одежде и обуви')
    article = attribute.get('артикул')
    brand_clean = clean(brand_1, no_brand, 2)

    if brand_clean is None:
        brand_clean = clean(brand_2, no_brand, 2)
    article_clean = clean(article, no_article, 3)
    partnumbers = ['артикул производителя', 'партномер (артикул производителя)', 'партномер']

    for partnumber in partnumbers:
        clean_partnumber= clean(attribute.get(partnumber), no_article, 3)
        if clean_partnumber is not None:
            break


    return brand_clean, article_clean, clean_partnumber


### Применение к обоим товарам пары


In [45]:
parsed_1 = pairs['attributes_1'].apply(parse_attributes).tolist()
df = pd.DataFrame(parsed_1, columns=['brand', 'article', 'partnumber'])
pairs['brand_1'] = df['brand']
pairs['article_1'] = df['article']
pairs['partnumber_1'] = df['partnumber']

parsed_2 = pairs['attributes_2'].apply(parse_attributes).tolist()
df = pd.DataFrame(parsed_2, columns=['brand', 'article', 'partnumber'])
pairs['brand_2'] = df['brand']
pairs['article_2'] = df['article']
pairs['partnumber_2'] = df['partnumber']

In [46]:
pairs.head()

,id1,id2,name_1,name_2,target,attributes_1,attributes_2,brand_1,article_1,partnumber_1,brand_2,article_2,partnumber_2
0,32,188978721454,очки с футляром-змейка fedrov мод. 1047 цвет 3...,фотохромные очки с футляром-змейка fedrov мод....,0.0,"{""диоптрии"":""- 2.00"",""цвет оправы"":""золотистый...","{""наличие флексов"":""да"",""затемнение"":""коричнев...",None,None,None,fedrov,None,None
1,34,197568543106,"зубная щетка r.o.c.s. black edition classic, ч...","зубная щетка r.o.c.s. red edition classic, сре...",0.0,"{""бренд"":""r.o.c.s."",""тип"":""зубная щетка"",""уров...","{""целевая аудитория"":""взрослая"",""бренд"":""r.o.c...",r.o.c.s.,None,None,r.o.c.s.,None,None
2,37,661425040668,сандалии fre gamo,fre gamo / сандалии,0.0,"{""бренд в одежде и обуви"":""fre gamo"",""страна б...","{""цвет"":""желтый"",""страна производства"":""китай""...",fre gamo,None,None,None,None,None
3,97,274877917548,acuvue астигматические линзы oasys for astigma...,acuvue / контактные линзы acuvue oasys for ast...,0.0,"{""ось (axis)"":""10"",""бренд"":""acuvue"",""цилиндр (...","{""состав"":""силикон-гидрогель"",""тип линз"":""тори...",acuvue,None,None,None,None,None
4,154,283467899529,"дворцовый пуэр (5 лет), 500 г",чайная линия ореховый пуэр (500 гр) вес: 500 гр,0.0,"{""бренд"":""weiserhouse"",""условия хранения"":""хра...","{""бренд"":""чайная линия"",""вид чая"":""пуэр"",""вкус...",weiserhouse,None,None,чайная линия,None,None


### Проверка извлечённых данных


In [47]:
pairs = pairs[['id1', 'id2', 'target',
               'name_1', 'name_2',
               'brand_1', 'brand_2',
               'article_1', 'article_2',
               'partnumber_1', 'partnumber_2']]

In [48]:
print(pairs)

                 id1           id2  target  \
0                 32  188978721454     0.0   
1                 34  197568543106     0.0   
2                 37  661425040668     0.0   
3                 97  274877917548     0.0   
4                154  283467899529     0.0   
...              ...           ...     ...   
365649  850403683693  695784731688     0.0   
365650  850403683737  369367295238     0.0   
365651  850403683764  377957271291     1.0   
365652  850403683771  231928288442     0.0   
365653  850403683774  171798796415     0.0   

                                                   name_1  \
0       очки с футляром-змейка fedrov мод. 1047 цвет 3...   
1       зубная щетка r.o.c.s. black edition classic, ч...   
2                                       сандалии fre gamo   
3       acuvue астигматические линзы oasys for astigma...   
4                           дворцовый пуэр (5 лет), 500 г   
...                                                   ...   
365649  alcon контак

In [49]:
print(pairs.notna().mean())

id1             1.000000
id2             1.000000
target          1.000000
name_1          1.000000
name_2          1.000000
brand_1         0.490940
brand_2         0.490499
article_1       0.132738
article_2       0.133993
partnumber_1    0.093616
partnumber_2    0.093744
dtype: float64


## 4. Словарь брендов и товары без бренда


In [50]:
pd.set_option('display.max_colwidth', None)

brands_1 = pairs[['id1', 'brand_1', 'name_1']].rename(
    columns={'id1': 'id', 'brand_1': 'brand', 'name_1': 'name'}
)
brands_2 = pairs[['id2', 'brand_2', 'name_2']].rename(
    columns={'id2': 'id', 'brand_2': 'brand', 'name_2': 'name'}
)

brands = pd.concat([brands_1, brands_2], ignore_index=True)
brands = brands.drop_duplicates(subset='id')
without_brands = brands[brands['brand'].isna()]
brands = brands.dropna()

brand_counts = brands['brand'].value_counts()

brands_one = brand_counts[brand_counts == 1]

brands[brands['brand'].isin(brands_one.index)].sample(20)


,id,brand,name
410616,558345791880,auplex,"auplex коврик для мыши коврики для мышки, красный, белый"
342019,798863992526,jojo print,jojo print сумка на плечо шоппер
327472,764504267133,alta nivelo,гель для наращивания и моделирования ногтей
726068,661425086235,eliodoro,пижама eliodoro
614949,309237649657,sacco,"закваска мезофильная для сыра lyofast mo 030 10uc (на 400-2000л, sacco)"
577323,34359770083,top4life,накладные ногти с клеем
189487,438086821525,vorel,отвертка крестовая рн1 x 150 мм
278617,652835082399,fortuna_kids,"не определен стеллаж для обуви, шимо светлый, тумба для обуви с мягким кожаным сиденьем, обувница в прихожую, полки для обуви"
448644,481036380383,kondylab,"обезвоженные съедобные засахаренные цветы, грейпфрут"
183418,429496780059,трика,рубашка трика абстракция


## 5. Поиск кандидатов в названиях


In [51]:
brands_sorted = sorted(brand_counts.index, key=len, reverse=True)
brands_pattern = '|'.join(re.escape(brand) for brand in brands_sorted)
brand_regex = re.compile(r'(?<!\w)(?:' + brands_pattern + r')(?!\w)')

def find_brand(name):
    found = brand_regex.findall(name.lower())
    return list(dict.fromkeys(found))




### Применение поиска и сохранение кандидатов


In [52]:
sample = without_brands.copy()
sample['brand_candidates'] = sample['name'].apply(find_brand)


In [53]:
sample['brand_candidates'].str.len().value_counts()


brand_candidates
1     188058
0      71724
2      71213
3      21924
4       5449
5       1306
6        593
7        197
8         30
9          9
10         2
Name: count, dtype: int64

In [54]:
sample.to_parquet('data/brand_candidates.parquet', index=False)

## 6. Присоединение кандидатов к парам


In [55]:
pairs_brands = pairs.merge(sample[['id', 'brand_candidates']], how='left', left_on='id1', right_on='id')

pairs_brands = pairs_brands.rename(
    columns={'brand_candidates': 'brand_candidates_1'}
)

pairs_brands = pairs_brands.drop(columns='id')

pairs_brands = pairs_brands.merge(sample[['id', 'brand_candidates']], how='left', left_on='id2', right_on='id')

pairs_brands = pairs_brands.rename(
    columns={'brand_candidates': 'brand_candidates_2'}
)

pairs_brands = pairs_brands.drop(columns='id')


## 7. Выбор откуда берем бренд


In [56]:
def get_brand_options(brand, candidates):
    if pd.notna(brand):
        return [brand]
    return candidates


In [57]:
pairs_brands['brands_1'] = pairs_brands.apply(
    lambda row: get_brand_options(row['brand_1'], row['brand_candidates_1']),
    axis=1
)

pairs_brands['brands_2'] = pairs_brands.apply(
    lambda row: get_brand_options(row['brand_2'], row['brand_candidates_2']),
    axis=1
)

## 8. Признак общего бренда


In [58]:
def has_common_brand(brands_1, brands_2):
    if not brands_1 or not brands_2:
        return None
    else:
        common = set(brands_1) & set(brands_2)
        return int(bool(common))


In [59]:
pairs_brands['brand_common'] = pairs_brands.apply(
    lambda row: has_common_brand(row['brands_1'], row['brands_2']),
    axis=1
)





In [60]:
pairs_brands['brand_common'].value_counts(dropna=False)

brand_common
1.0    218550
0.0     87179
NaN     59925
Name: count, dtype: int64

In [61]:
pairs_brands

,id1,id2,target,name_1,name_2,brand_1,brand_2,article_1,article_2,partnumber_1,partnumber_2,brand_candidates_1,brand_candidates_2,brands_1,brands_2,brand_common
0,32,188978721454,0.0,очки с футляром-змейка fedrov мод. 1047 цвет 3 с линзами romeo 1.56 hmc/emi -2.00 рц 62-64,"фотохромные очки с футляром-змейка fedrov мод. 1047 цвет 3 с линзами romeo 1.56 fast photocolor brown, hmc+ -3.00 рц 64-66",None,fedrov,None,None,None,None,"[fedrov, romeo, emi]",NaN,"[fedrov, romeo, emi]",[fedrov],1.0
1,34,197568543106,0.0,"зубная щетка r.o.c.s. black edition classic, черная/красная","зубная щетка r.o.c.s. red edition classic, средняя",r.o.c.s.,r.o.c.s.,None,None,None,None,NaN,NaN,[r.o.c.s.],[r.o.c.s.],1.0
2,37,661425040668,0.0,сандалии fre gamo,fre gamo / сандалии,fre gamo,None,None,None,None,None,NaN,[fre gamo],[fre gamo],[fre gamo],1.0
3,97,274877917548,0.0,"acuvue астигматические линзы oasys for astigmatism,ось:10_-0.75, 6 шт., -0.75 / 8.6/-0.75/10, 2 недели",acuvue / контактные линзы acuvue oasys for astigmatism (6 линз) r 8.6 d +4.75 cyl -0.75 ax 10,acuvue,None,None,None,None,None,NaN,"[acuvue, acuvue oasys]",[acuvue],"[acuvue, acuvue oasys]",1.0
4,154,283467899529,0.0,"дворцовый пуэр (5 лет), 500 г",чайная линия ореховый пуэр (500 гр) вес: 500 гр,weiserhouse,чайная линия,None,None,None,None,NaN,NaN,[weiserhouse],[чайная линия],0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
365649,850403683693,695784731688,0.0,"alcon контактные линзы air optix night & day aqua, 3 шт. / 8.6 / 1 месяц, ежемесячные / +02.50","контактные линзы alcon air optix night & day aqua, 3 шт., r 8,6, d -9",alcon,None,None,None,None,None,NaN,"[alcon, air optix, aqua]",[alcon],"[alcon, air optix, aqua]",1.0
365650,850403683737,369367295238,0.0,shakmy / приправа для мяса 1 кг.,"приправа для мяса, 1кг",None,carp box,None,None,None,None,[],NaN,[],[carp box],NaN
365651,850403683764,377957271291,1.0,"3d art.игрушка-раскраска ""снежный барс""","lori / набор для творчества раскраска игрушка "" 93335",lori,None,None,None,None,None,NaN,"[lori, игрушка]",[lori],"[lori, игрушка]",1.0
365652,850403683771,231928288442,0.0,"грузовик радиоуправляемый самосвал, работает от батареек,","грузовик радиоуправляемый «самосвал», работает от батареек,",None,None,None,None,None,None,[],[],[],[],NaN


## 9. Сохранение результатов


In [62]:
pairs_brands[['id1', 'id2', 'target', 'brand_common']].to_parquet(
    'data/brand_features.parquet',
    index=False
)


In [63]:
pairs.to_parquet('data/pairs_attrs.parquet')

признак пары `brand_common`. Для каждого товара используется бренд из характеристик, иначе кандидаты, найденные в названии по словарю брендов.

1 - есть хотя бы один общий бренд </br>
0 - информация есть у обоих товаров, но общих брендов нет </br>
NaN - хотя бы у одного товара нет информации о бренде </br>


In [64]:
pairs_attrs = pd.read_parquet('data/pairs_attrs.parquet')


In [65]:
def compare_attribute(value_1, value_2):
    if pd.isna(value_1) or pd.isna(value_2):
        return None
    if value_1 == value_2:
        return 1
    return 0

pairs_attrs['article_equal'] = pairs_attrs.apply(
    lambda row: compare_attribute(row['article_1'], row['article_2']),
    axis=1
)

pairs_attrs['partnumber_equal'] = pairs_attrs.apply(
    lambda row: compare_attribute(row['partnumber_1'], row['partnumber_2']),
    axis=1
)

pairs_attrs['brand_equal'] = pairs_attrs.apply(
    lambda row: compare_attribute(row['brand_1'], row['brand_2']),
    axis=1
)


In [66]:
pairs_attrs["article_equal"].value_counts(dropna=False)


article_equal
NaN    363647
0.0      1981
1.0        26
Name: count, dtype: int64

In [67]:
pairs_attrs["partnumber_equal"].value_counts(dropna=False)

partnumber_equal
NaN    355834
0.0      8906
1.0       914
Name: count, dtype: int64

In [68]:
pairs_attrs['brand_equal'].value_counts(dropna=False)

brand_equal
NaN    322168
1.0     29380
0.0     14106
Name: count, dtype: int64

In [69]:
pairs_attrs.to_parquet('data/pairs_attrs.parquet', index=False)

In [70]:
pairs_for_type = pd.read_parquet('data/pairs.parquet', columns=['id1', 'id2', 'attributes_1', 'attributes_2', 'target'])



In [71]:
pairs_for_type.head()

,id1,id2,attributes_1,attributes_2,target
0,32,188978721454,"{""диоптрии"":""- 2.00"",""цвет оправы"":""золотистый, коричневый"",""назначение"":""корректирующие"",""конструкция"":""ободковая"",""материал оправы"":""комбинированный"",""материал линз"":""пластик"",""форма оправы"":""бабочка"",""покрытие линз"":""упрочняющее, водоотталкивающее и антистатическое, металлизированное (защита от электромагнитных волн) uv-380, просветляющее"",""пол"":""женские"",""ширина линзы"":""56 мм"",""высота линзы"":""52 мм"",""межцентровое расстояние"":""62-64"",""длина заушника"":""144 мм""}","{""наличие флексов"":""да"",""затемнение"":""коричневый фотохром 80%"",""бренд"":""fedrov"",""цвет товара"":""золотой"",""название цвета"":""brown & gold"",""страна-изготовитель"":""россия"",""размер заушника, мм"":""144"",""коллекция (год)"":""2022"",""конструкция очков"":""ободковые"",""форма оправы"":""бабочки"",""материал оправы очков"":""комбинированный"",""материал линз"":""пластик"",""стиль оправы"":""повседневный"",""форма лица"":""квадратная"",""покрытие линз"":""упрочняющее, просветляющее, металлизированное (защита от электромагнитных волн) emi, uv400, водоотталкивающее и антистатическое"",""высота линзы, мм"":""52"",""ширина линзы, мм"":""56"",""размер моста на переносице, мм"":""15"",""расстояние между оптическими центрами"":""64-66"",""оптическая сила"":""-3.00"",""комплектация"":""очки fedrov с полимерными линзами romeo photocolor brown 1.56 hmc+, микрофибровая салфетка для ухода, футляр-змейка."",""пол ребенка"":""унисекс"",""пол"":""женский"",""общая ширина, мм"":""139"",""тип"":""очки для дали""}",0.0
1,34,197568543106,"{""бренд"":""r.o.c.s."",""тип"":""зубная щетка"",""уровень жесткости"":""средняя"",""особенности"":""нескользящая ручка"",""минимальный возраст"":""от 16 лет"",""материал щетины"":""синтетическая"",""материал корпуса"":""пластик"",""длина головки"":""18.4 мм"",""способ применения"":""скошенная подстрижка щетины позволяет зубной щётке проникать в труднодоступные места и к дальним зубам. благодаря технологии тройной полировки кончиков щетинки идеально закруглённые и гладкие, поэтому безопасны для дёсен.тонкая лёгкая ручка из безопасного высококачественного пластика предотвращает излишнее давление при чистике. кроме того, её гладкая поверхность без дополнительных деталей препятствует размножению бактерий, что обеспечивает гигиеничность щётки."",""дополнительная информация"":""скошенная подстрижка щетины позволяет зубной щётке проникать в труднодоступные места и к дальним зубам. благодаря технологии тройной полировки кончиков щетинки идеально закруглённые и гладкие, поэтому безопасны для дёсен.тонкая лёгкая ручка из безопасного высококачественного пластика предотвращает излишнее давление при чистике. кроме того, её гладкая поверхность без дополнительных деталей препятствует размножению бактерий, что обеспечивает гигиеничность щётки"",""цвет товара"":""черная/красная"",""количество упаковок"":""другое""}","{""целевая аудитория"":""взрослая"",""бренд"":""r.o.c.s."",""цвет товара"":""красный"",""страна-изготовитель"":""россия"",""жесткость щетины"":""средняя"",""тип"":""зубная щетка""}",0.0
2,37,661425040668,"{""бренд в одежде и обуви"":""fre gamo"",""страна бренда"":""россия"",""коллекция"":""весна-лето 2021"",""цвет товара"":""черный"",""название цвета"":""черный"",""страна-изготовитель"":""китай"",""материал верха"":""искусственная кожа"",""материал"":""искусственная кожа"",""российский размер (обуви)"":""43"",""сезон"":""лето"",""пол"":""мужской"",""размер производителя"":""43"",""материал подошвы"":""резина""}","{""цвет"":""желтый"",""страна производства"":""китай"",""комплектация"":""1 пара"",""валюта"":""rub"",""российский размер"":""41""}",0.0
3,97,274877917548,"{""ось (axis)"":""10"",""бренд"":""acuvue"",""цилиндр (cyl)"":""-0.75"",""страна-изготовитель"":""сша"",""диаметр, мм"":""14.5"",""особенности контактных линз"":""уф-фильтр"",""радиус кривизны"":""8.6"",""материал контактных линз"":""этафилкон а"",""количество в у

In [85]:
def extract_attribute(attribute, field):
    attribute = json.loads(attribute)
    value = attribute.get(field)
    value = clean(value, [])
    return value


def clean(value, bad_values, min_len=1):
    if value is None:
        return None
    value = value.lower().strip()
    if value == '' or value in bad_values or len(value) < min_len:
        return None
    return value

def compare_attribute(value_1, value_2, type='equal'):
    if pd.isna(value_1) or pd.isna(value_2):
        return None
    if type == 'ratio':
        return fuzz.ratio(value_1, value_2)
    else:
        if value_1 == value_2:
            return 1
        return 0



def add_attribute_columns(pairs, field, name):
    pairs[name + '_1'] = pairs['attributes_1'].apply(extract_attribute, field=field)
    pairs[name + '_2'] = pairs['attributes_2'].apply(extract_attribute, field=field)


add_attribute_columns(pairs_for_type, 'тип', 'type')
pairs_for_type['type_similarity'] = pairs_for_type.apply(
    lambda row: compare_attribute(row['type_1'], row['type_2'], type='ratio'),
    axis=1
)


add_attribute_columns(pairs_for_type, 'цвет товара', 'color')
pairs_for_type['color_equal'] =  pairs_for_type.apply(
    lambda row: compare_attribute(row['color_1'], row['color_2']), axis=1
)

add_attribute_columns(pairs_for_type, 'материал', 'material')
pairs_for_type['material_equal'] =  pairs_for_type.apply(
    lambda row: compare_attribute(row['material_1'], row['material_2']), axis=1
)

add_attribute_columns(pairs_for_type, 'российский размер', 'size')
pairs_for_type['size_equal'] =  pairs_for_type.apply(
    lambda row: compare_attribute(row['size_1'], row['size_2']), axis=1
)


In [86]:
mean_type_similarity = pairs_for_type['type_similarity'].mean()
type_similarity_counts = pairs_for_type['type_similarity'].value_counts(dropna=False)

color_match_rate = pairs_for_type['color_equal'].mean()
color_equal_counts = pairs_for_type['color_equal'].value_counts(dropna=False)

material_match_rate = pairs_for_type['material_equal'].mean()
material_equal_counts = pairs_for_type['material_equal'].value_counts(dropna=False)

size_match_rate = pairs_for_type['size_equal'].mean()
size_equal_counts = pairs_for_type['size_equal'].value_counts(dropna=False)

print(size_match_rate, size_equal_counts)

0.07446424705967242 size_equal
NaN    351795
0.0     12827
1.0      1032
Name: count, dtype: int64


In [78]:
pairs_for_type[pairs_for_type['color_equal'] == 0]

,id1,id2,attributes_1,attributes_2,target,type_1,type_2,type_similarity,color_1,color_2,color_equal
1,34,197568543106,"{""бренд"":""r.o.c.s."",""тип"":""зубная щетка"",""уровень жесткости"":""средняя"",""особенности"":""нескользящая ручка"",""минимальный возраст"":""от 16 лет"",""материал щетины"":""синтетическая"",""материал корпуса"":""пластик"",""длина головки"":""18.4 мм"",""способ применения"":""скошенная подстрижка щетины позволяет зубной щётке проникать в труднодоступные места и к дальним зубам. благодаря технологии тройной полировки кончиков щетинки идеально закруглённые и гладкие, поэтому безопасны для дёсен.тонкая лёгкая ручка из безопасного высококачественного пластика предотвращает излишнее давление при чистике. кроме того, её гладкая поверхность без дополнительных деталей препятствует размножению бактерий, что обеспечивает гигиеничность щётки."",""дополнительная информация"":""скошенная подстрижка щетины позволяет зубной щётке проникать в труднодоступные места и к дальним зубам. благодаря технологии тройной полировки кончиков щетинки идеально закруглённые и гладкие, поэтому безопасны для дёсен.тонкая лёгкая ручка из безопасного высококачественного пластика предотвращает излишнее давление при чистике. кроме того, её гладкая поверхность без дополнительных деталей препятствует размножению бактерий, что обеспечивает гигиеничность щётки"",""цвет товара"":""черная/красная"",""количество упаковок"":""другое""}","{""целевая аудитория"":""взрослая"",""бренд"":""r.o.c.s."",""цвет товара"":""красный"",""страна-изготовитель"":""россия"",""жесткость щетины"":""средняя"",""тип"":""зубная щетка""}",0.0,зубная щетка,зубная щетка,100.0,черная/красная,красный,0.0
19,963,214748390349,"{""материал обивки"":""искусственная кожа"",""цвет товара"":""черный"",""назначение"":""для руководителя"",""конструктивные особенности"":""с газлифтом, с колесами (роликами), с подголовником, с подлокотниками, с поясничным упором"",""функциональные особенности"":""механизм качания, на колесиках, ортопедические свойства, повышенная прочность, регулировка по высоте, с подголовником, с подлокотниками"",""максимальная нагрузка"":""120 кг"",""регулировка"":""высоты сиденья, жесткости качания, наклона спинки"",""тип механизма качания"":""топ-ган"",""количество положений фиксации спинки"":""1"",""комплектация"":""спинка кресла-1 шт; сиденье-1 шт; крестовина-1 шт; колесики-5 шт, газ-лифт-1 шт; подлокотники-2 шт; болты-8 шт; шестигранник-1 шт; заглушки 8 шт; инструкция-1 шт"",""страна бренда"":""россия"",""материал каркаса"":""мебельная фанера"",""материал крестовины"":""пластик"",""материал накладки на крестовину"":""искусственная кожа"",""материал подлокотников"":""пластик"",""материал накладок на подлокотники"":""пластик"",""материал наполнителя"":""поролон"",""высота кресла"":""113 см"",""максимальная высота кресла"":""127 см"",""высота сиденья"":""48 см"",""максимальная высота сиденья"":""60 см"",""глубина сиденья"":""47.5 см"",""максимальная глубина сиденья"":""47.5 см"",""ширина сиденья"":""47 см"",""высота спинки"":""70 см"",""вес"":""14 кг"",""габариты транспортной упаковки"":""83.50х40х63.50 см"",""вес в транспортной упаковке"":""14 кг""}","{""артикул"":""00-00095800"",""ширина спинки, см"":""32"",""бренд"":""нет бренда"",""цвет товара"":""разноцветный"",""страна-изготовитель"":""россия"",""глубина, см"":""32"",""форма поставки"":""в разобранном виде"",""высота, см"":""56"",""вид выпуска товара"":""фабричное производство"",""материал обивки"":""без обивки"",""материал корпуса"":""abs пластик"",""комплектация"":""кресло vt-сhairman 205 (prestige ergo) ткань черная"",""стиль дизайна"":""классический"",""тип"":""кресло"",""гарантийный срок"":""365"",""вес товара, г"":""8200"",""ширина, см"":""32"",""количество в комплекте, шт."":""1""}",0.0,None,кресло,NaN,черный,разноцветный,0.0
24,1192,549755853280,"{""бренд"":""пижон"",""вид ошейника"":""повседневный"",""ширина ошейника, мм"":""3"",""цвет товара"":""синий"",""название цвета"":""синий"",""страна-изгот

In [74]:
pairs_for_type[pairs_for_type['type_similarity'] == 0]


,id1,id2,attributes_1,attributes_2,target,type_1,type_2,type_similarity,color_1,color_2,color_equal
348,15884,738734488744,"{""цвет товара"":""серебристый"",""тип"":""чашка"",""объем"":""70 мл"",""высота"":""5.5 см"",""диаметр"":""5.5 см"",""материал"":""серебро"",""праздник"":""универсальный""}","{""целевая аудитория"":""взрослая"",""артикул"":""1144нб07806"",""бренд"":""аргента"",""коллекция"":""базовая коллекция"",""цвет товара"":""серебристый"",""страна-изготовитель"":""россия"",""размеры, мм"":""чашка высота 55 мм, диаметр 55 мм. объем 70 мл."",""тип ювелирного сувенира"":""другое"",""проба"":""925"",""комплектация"":""состав набора :\n1. чашка кофейная \""роза\"" черн. нового образца арт. 1056чш07006 - 2шт. (средний вес 2 х 75 гр. = 150 гр.) длина: 55 мм. ширина: 55 мм. высота: 55 мм. диаметр: 55 мм. объём: 70 мл.\n2. ложка чайная \""роза\"" литая с черн. арт. 1079лж03006 - 2шт. (средний вес 2 х 21 гр. = 42 гр.) длина: 125 мм. высота: 10 мм.\n3. футляр арт. ф1970407 - 1шт. длина: 235 мм. ширина: 175 мм. высота: 60 мм.\nсредний вес: 192 гр\nфутляр или коробка может отличаться от заявленных на фото. "",""упаковка"":""подарочный футляр"",""тематика украшения"":""цветы и растения"",""тип"":""сувенир ювелирный"",""вес товара, г"":""192"",""средний вес изделия, гр"":""192"",""состав ювелирного изделия"":""серебро""}",0.0,чашка,сувенир ювелирный,0.0,серебристый,серебристый,1.0
731,31246,283467990824,"{""цвет товара"":""бежевый"",""тип"":""-"",""назначение"":""для акустической гитары, для классической гитары, для электрогитары"",""материал"":""нейлон""}","{""бренд"":""cort"",""тип"":""акустическая гитара""}",0.0,-,акустическая гитара,0.0,бежевый,None,NaN
1272,55288,77309500315,"{""материал корпуса/верхней деки"":""липа"",""бренд"":""aria"",""тип бриджа"":""тремоло"",""цвет товара"":""красный"",""название цвета"":""красный"",""страна-изготовитель"":""китай"",""материал накладки грифа"":""палисандр"",""ширина грифа, мм"":""43"",""количество ладов, шт."":""24"",""форма гитары"":""jazzmaster/jaguar"",""размер инструмента"":""4/4"",""мензура, мм"":""648"",""материал грифа"":""клен"",""крепление грифа"":""на болтах"",""комплектация"":""гитара"",""наличие звукоснимателя"":""true"",""звукосниматели"":""os-5 (alnico-5) & cph-1 (classic power alnico-5)"",""материал струн"":""металл"",""количество струн"":""6"",""тип"":""электрогитара""}","{""тип"":""-"",""мензура"":""24.7244\"""",""количество струн"":""6"",""тип крепления грифа"":""болченый"",""форма корпуса электрогитары"":""stratocaster"",""материал корпуса"":""тополь"",""тип бриджа"":""тремоло"",""количество ладов"":""24"",""инкрустация"":""в виде точек"",""цвет фурнитуры"":""хром""}",1.0,электрогитара,-,0.0,красный,None,NaN
1879,82890,197568515022,"{""бренд"":""нет бренда"",""форм-фактор"":""2.5\"""",""тип накопителя"":""ssd"",""материал корпуса"":""металл"",""комплектация"":""переходник optibay, отвертка, винты."",""тип"":""адаптер для ssd/hdd"",""гарантийный срок"":""1 месяц"",""вес товара, г"":""95""}","{""тип"":""optibay"",""форм-фактор"":""2.5\"""",""тип охлаждения"":""пассивное"",""материал корпуса"":""пластик/металл"",""гарантийный срок"":""3 мес., гарантия — 3 мес.""}",0.0,адаптер для ssd/hdd,optibay,0.0,None,None,NaN
2999,131194,420906861133,"{""бренд"":""netac"",""цвет товара"":""черный"",""страна-изготовитель"":""китай"",""размеры, мм"":""100х29х9"",""форм-фактор"":""1.8\"""",""комплектация"":""кабель usb type-a, переходник usb type-a - usb type-c"",""скорость чтения, мб/с"":""550"",""скорость записи, мб/с"":""480"",""вид накопителя"":""портативный"",""тип"":""внешний ssd-диск"",""объем"":""128 гб"",""гарантийный срок"":""12 месяцев""}","{""тип"":""36777240"",""форм-фактор"":""1.8\"""",""гарантийный срок"":""1 г.""}",0.0,внешний ssd-диск,36777240,0.0,черный,None,NaN
...,...,...,...,...,...,...,...,...,...,...,...
363556,850403591433,292057915015,"{""тип"":""-"",""тип корпуса"":""drednought"",""размер"":""4/4"",""с вырезом"":""да"",""количество ладов"":""20"",""мензура"":""6

In [88]:
pairs_attrs = pd.read_parquet('data/pairs_attrs.parquet')

pairs_attrs = pairs_attrs.drop(columns=['type_similarity', 'color_equal'], errors='ignore')

pairs_attrs = pairs_attrs.merge(
    pairs_for_type[['id1', 'id2', 'type_similarity', 'color_equal', 'material_equal', 'size_equal']],
    on=['id1', 'id2'],
    how='left',
    validate='one_to_one'
)

pairs_attrs.to_parquet('data/pairs_attrs.parquet', index=False)

